# 03b · De-Identified Silver → Safe Gold Star Schema

> **SYNTHETIC DATA ONLY.** Reference pattern — not a certified de-identification service.

Builds the **PHI-free** `gold_safe_*` star schema from `silver_deid_*`. This is the layer
Power BI and Copilot point at — and by construction it contains **zero** HIPAA Safe Harbor
identifiers.

Compare with `03_gold_star.ipynb` (the **"before"**): that one carries raw `MRN`,
`PatientName`, `DateOfBirth`, and full `ZIP` all the way into Gold. Toggling the semantic
model between `gold_*` and `gold_safe_*` is the live demo money-shot.

**Date grain:** under Safe Harbor the grain is **Year** (no daily `Dim_Date`). Under Expert
Determination the shifted dates in `silver_deid_*` can rebuild a daily date dimension.

In [ ]:
from pyspark.sql import functions as F
import sys, os

# --- make the accelerator's package importable ---
# This notebook does NOT build Spark UDFs (pure column select + native year()), and the
# residual-PHI scan collects to the driver and scans in Python. So a driver-side sys.path
# is sufficient here — no addPyFile needed. SRC_PATH is the PARENT of fabric_phi_deid/.
SRC_PATH    = "/lakehouse/default/Files/accelerator"
CONFIG_PATH = "/lakehouse/default/Files/accelerator/config/deid_rules.yaml"
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

from fabric_phi_deid.deid_engine import load_rules            # noqa: E402
from fabric_phi_deid.validation import scan_spark_dataframe   # noqa: E402
from fabric_phi_deid.audit import get_audit_logger            # noqa: E402

log = get_audit_logger()
PROFILE = load_rules(CONFIG_PATH).get("active_profile", "safe_harbor")
DEID, G = "silver_deid_", "gold_safe_"

def deid(t): return spark.table(f"{DEID}{t}")

def year_of(colname):
    """Return a Year int whether the de-id column is an int-year (Safe Harbor) or a
    shifted date (Expert Determination)."""
    c = F.col(colname)
    return c.cast("int") if PROFILE == "safe_harbor" else F.year(c.cast("date"))

# --- fail-fast: gold_safe_* is built ONLY from de-identified silver; those must exist ---
_required = [
    "dim_patient", "dim_provider", "dim_department", "dim_facility", "dim_payer",
    "dim_diagnosis", "dim_procedure", "fact_encounter", "fact_claim", "fact_risk_score",
]
_existing = {t.name for t in spark.catalog.listTables()}
_missing  = [f"{DEID}{t}" for t in _required if f"{DEID}{t}" not in _existing]
if _missing:
    raise RuntimeError(
        "Cannot build gold_safe_*: missing de-identified silver tables: "
        + ", ".join(_missing)
        + ".\nRun 02b_silver_deid first — it is the single privileged crossing point."
    )
if not os.path.isfile(CONFIG_PATH):
    raise RuntimeError(f"Missing de-id rulebook at {CONFIG_PATH} — upload the accelerator config first.")

print(f"Profile: {PROFILE}  |  silver_deid_* prerequisites present: {len(_required)}/{len(_required)}")


In [ ]:
# ---------- conformed dimensions (PHI-free) ----------
# DateOfBirth is a birth YEAR (int) under Safe Harbor -> expose as BirthYear.
gold_dims = {
    "dim_patient": deid("dim_patient").select(
        "PatientKey", "MRN", "PatientName", year_of("DateOfBirth").alias("BirthYear"),
        "Age", "AgeBand", "Gender", "Race", "Ethnicity", "ZIP", "PCPProviderKey"),
    "dim_provider": deid("dim_provider").select(
        "ProviderKey", "NPI", "ProviderFullName", "Credentials", "Gender",
        "ProviderType", "PrimarySpecialty", "ProviderStatus", "IsActive",
        "HireDate", "TerminationDate"),
    "dim_department": deid("dim_department").select(
        "DepartmentKey", "DepartmentName", "ServiceLine", "DepartmentSpecialty"),
    "dim_facility": deid("dim_facility").select(
        "FacilityKey", "FacilityName", "City", "StateAbbr", "ZIP", "Region"),
    "dim_payer": deid("dim_payer").select(
        "PayerKey", "PayerName", "PlanType", "LineOfBusiness"),
    "dim_diagnosis": deid("dim_diagnosis").select(
        "DiagnosisKey", "ICD10Code", "DiagnosisName", "DiagnosisCategory"),
    "dim_procedure": deid("dim_procedure").select(
        "ProcedureKey", "ProcedureCode", "CodeType", "ProcedureDescription"),
}
for name, df in gold_dims.items():
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{G}{name}"))
    print(f"{G+name:28s} {df.count():>8,} rows")

In [ ]:
# ---------- fact tables (Year grain instead of a daily DateKey) ----------
fact_encounter = deid("fact_encounter").select(
    "EncounterKey", "PatientKey", "AttendingProviderKey", "ReferringProviderKey",
    "DepartmentKey", "LocationKey", "DiagnosisKey", "EncounterType", "LengthOfStay",
    year_of("EncounterDate").alias("EncounterYear"))

fact_claim = deid("fact_claim").select(
    "ClaimKey", "PatientKey", "BillingProviderKey", "RenderingProviderKey", "PayerKey",
    "ProcedureKey", "DiagnosisKey", "BilledAmount", "AllowedAmount", "PaidAmount",
    "PatientResponsibility", "AllowedVariance", "ClaimStatus", "DeniedFlag",
    year_of("ServiceDate").alias("ServiceYear"))

fact_risk = deid("fact_risk_score").select(
    "RiskScoreKey", "PatientKey", "ProviderKey", "RiskModel", "RiskScore",
    year_of("ScoreDate").alias("ScoreYear"))

for name, df in {"fact_encounter": fact_encounter, "fact_claim": fact_claim,
                 "fact_risk_score": fact_risk}.items():
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{G}{name}"))
    print(f"{G+name:28s} {df.count():>8,} rows")

print("\nGold tables written. Running the publish gate before declaring them safe...")


In [ ]:
# ---------- PUBLISH GATE: prove gold_safe_* is PHI-free BEFORE Power BI / Copilot see it ----------
# The header promises "zero HIPAA Safe Harbor identifiers." Don't trust the upstream de-id —
# verify the PUBLISHED layer directly. This is defense-in-depth on top of 02b's silver gate:
# a value that slipped through silver would be caught here, before anything is shared.
_gold_tables = [
    "dim_patient", "dim_provider", "dim_department", "dim_facility", "dim_payer",
    "dim_diagnosis", "dim_procedure", "fact_encounter", "fact_claim", "fact_risk_score",
]

# 1) residual direct-identifier scan (SSN / phone / email) across all gold string columns
_residual = {}
for t in _gold_tables:
    hits = scan_spark_dataframe(spark.table(f"{G}{t}"), sample_limit=10000)
    if hits:
        _residual[f"{G}{t}"] = hits

# 2) row-count reconciliation — gold is a projection of silver_deid; counts must match exactly
_recon = []
print(f"{'table':24s} {'silver_deid':>12s} {'gold_safe':>12s}  status")
for t in _gold_tables:
    s, g = deid(t).count(), spark.table(f"{G}{t}").count()
    status = "OK" if s == g else "MISMATCH"
    _recon.append((t, s, g, status))
    print(f"{t:24s} {s:>12,} {g:>12,}  {status}")
_bad = [r for r in _recon if r[3] != "OK"]

# 3) block publication on any failure (fail closed)
if _residual:
    log.error(f"gold publish BLOCKED — residual PHI in {list(_residual)}")
    raise AssertionError(f"Residual PHI detected in gold_safe_*: {_residual}")
if _bad:
    log.error(f"gold publish BLOCKED — row-count mismatch: {_bad}")
    raise AssertionError(f"Row-count mismatch silver_deid -> gold_safe: {_bad}")

log.info(f"gold_safe_* published clean: {len(_gold_tables)} tables, 0 residual PHI, counts reconciled")
print("\nPUBLISH GATE PASSED — 0 residual PHI, all row counts reconciled.")
print("Safe to point Power BI / Copilot at gold_safe_*.  Deeper validation: NB_scorecard.")
